In [1]:
#apply StandardScaler
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
# Load dataset
df= pd.read_csv("data/AmesHousing_engineered.csv")

# Drop target and ID columns
X = df.drop(columns=["SalePrice", "PID", "Order"], errors="ignore")
print("Features shape (scaled version):", X.shape)

Features shape (scaled version): (2930, 172)


In [2]:
#Apply StandardScaler
#Now X_scaled contains all features standardized (mean = 0, std = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features shape (scaled version):", X_scaled.shape)

Features shape (scaled version): (2930, 172)


In [3]:
#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [4]:
#K-Means on Scaled Data
start_time = time.time()
kmean_scaled = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    kmean_scaled.append({"algorithm": "KMeans", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 6.603492736816406 seconds
K-Means runtime: 6.6035 seconds


In [5]:
#Gaussian Mixture (GMM)on Scaled Data
start_time = time.time()
gmm_scaled = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    gmm_scaled.append({"algorithm": "GMM", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")

Runtime: 195.63233590126038 seconds
GMM runtime: 195.6323 seconds


In [6]:
#Agglomerative Clustering on Scaled Data
start_time = time.time()
agg_scaled = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    agg_scaled.append({"algorithm": "Agglomerative", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Runtime: 7.639333248138428 seconds
Agglomerative runtime: 7.6393 seconds


In [7]:
#Spectral Clustering on Scaled Data
start_time = time.time()
spec_scaled = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    spec_scaled.append({"algorithm": "Spectral", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 8.43029260635376 seconds
Spectral runtime: 8.4303 seconds


In [8]:
#DBSCAN on Scaled Data
start_time = time.time()
dbscan_scaled = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_scaled)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 :  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_scaled[mask], labels[mask])
        dbscan_scaled.append({"algorithm": "DBSCAN", "preprocessing": "Scaled", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Dbscan runtime: {runtime:.4f} seconds")

Runtime: 0.17796754837036133 seconds
Dbscan runtime: 0.1780 seconds


In [9]:

from sklearn.cluster import Birch
start_time = time.time()
birch_scaled = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    #birch = Birch(n_clusters=4, threshold=t)
    labels = birch.fit_predict(X_scaled)

    if len(set(labels)) > 1:
        sil, db, ch = compute_metrics(X_scaled, labels)
        birch_scaled.append({
            "algorithm": "BIRCH",
            "preprocessing": "Scaled",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Birch runtime: {runtime:.4f} seconds")
print(birch_scaled)

Runtime: 9.973183155059814 seconds
Birch runtime: 9.9732 seconds
[{'algorithm': 'BIRCH', 'preprocessing': 'Scaled', 'threshold': 0.2, 'n_clusters': 2919, 'silhouette': 0.004666489292569091, 'davies_bouldin': 0.027254070476445527, 'calinski_harabasz': 1293.2745481875104}, {'algorithm': 'BIRCH', 'preprocessing': 'Scaled', 'threshold': 0.5, 'n_clusters': 2898, 'silhouette': 0.011830262253272964, 'davies_bouldin': 0.08544021247962665, 'calinski_harabasz': 143.44633493027828}, {'algorithm': 'BIRCH', 'preprocessing': 'Scaled', 'threshold': 1.0, 'n_clusters': 2788, 'silhouette': 0.02898420846935497, 'davies_bouldin': 0.24446880264784213, 'calinski_harabasz': 29.293102891041414}, {'algorithm': 'BIRCH', 'preprocessing': 'Scaled', 'threshold': 1.5, 'n_clusters': 2437, 'silhouette': 0.038416441995701, 'davies_bouldin': 0.4522112037226535, 'calinski_harabasz': 13.217343761920736}]


In [10]:
from sklearn.cluster import OPTICS
start_time = time.time()
optics_scaled = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_scaled)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_scaled, labels)
        optics_scaled.append({
            "algorithm": "OPTICS",
            "preprocessing": "Scaled",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

Runtime: 85.34103226661682 seconds
Optics runtime: 85.3410 seconds


In [11]:
import csv


ames_results_scaled = (kmean_scaled + gmm_scaled + agg_scaled + spec_scaled + dbscan_scaled+birch_scaled + optics_scaled)
# Desired column order
keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/ames_data/ames_scaled.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(ames_results_scaled)

In [11]:
from sklearn.metrics import adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

# ARI stability analysis
n_bootstrap = 100
ari_results = []
# Collect all parameter settings from your previous results
all_configs = []

for r in kmean_scaled:
    all_configs.append(("K-Means", {"k": r["k"]}))

for r in gmm_scaled:
    all_configs.append(("GMM", {"k": r["k"]}))

for r in agg_scaled:
    all_configs.append(("Agglomerative", {"k": r["k"]}))

for r in spec_scaled:
    all_configs.append(("Spectral", {"k": r["k"]}))

for r in dbscan_scaled:
    all_configs.append(("DBSCAN", {"eps": r["eps"]}))

for r in birch_scaled:
    all_configs.append(("BIRCH", {"threshold": r["threshold"]}))

for r in optics_scaled:
    all_configs.append(("OPTICS", {"min_samples": r["min_samples"]}))
# helper function to fit a model and return labels 
def fit_and_predict(name, params, X_data):

    if name == "K-Means":
        model = KMeans(n_clusters=params["k"], n_init=n_init, random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "GMM":
        model = GaussianMixture(n_components=params["k"], n_init=n_init, random_state=42)
        labels = model.fit(X_data).predict(X_data)

    elif name == "Agglomerative":
        model = AgglomerativeClustering(n_clusters=params["k"], linkage='ward')
        labels = model.fit_predict(X_data)

    elif name == "Spectral":
        model = SpectralClustering(
            n_clusters=params["k"],
            affinity='nearest_neighbors',
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "DBSCAN":
        model = DBSCAN(eps=params["eps"], min_samples=min_samples)
        labels = model.fit_predict(X_data)

    elif name == "BIRCH":
        model = Birch(n_clusters=None, threshold=params["threshold"])
        labels = model.fit_predict(X_data)

    elif name == "OPTICS":
        model = OPTICS(min_samples=params["min_samples"], xi=0.05, n_jobs=-1)
        labels = model.fit_predict(X_data)

    else:
        return None

    return labels


# Reuse all parameter configurations from previous section
for algo_name, params in all_configs:

    # reference clustering on full data
    ref_labels = fit_and_predict(algo_name, params, X_scaled)

    if ref_labels is None:
        continue

    ari_scores = []
    rng = np.random.RandomState(42)

    for b in range(n_bootstrap):

        # bootstrap sample with indices
        indices = rng.choice(len(X_scaled), size=len(X_scaled), replace=True)
        X_boot = X_scaled[indices]

        boot_labels = fit_and_predict(algo_name, params, X_boot)

        if boot_labels is None:
            continue

        # compare only sampled observations
        ref_subset = np.array(ref_labels)[indices]

        # remove noise points for DBSCAN / OPTICS
        mask = (boot_labels != -1) & (ref_subset != -1)

        if np.sum(mask) < 2:
            continue

        ari = adjusted_rand_score(ref_subset[mask], np.array(boot_labels)[mask])
        ari_scores.append(ari)

    if len(ari_scores) > 0:
        ari_results.append({
            "algorithm": algo_name,
            **params,
            "ARI_mean": np.mean(ari_scores),
            "ARI_std": np.std(ari_scores)
        })


# Summary table
ari_df = pd.DataFrame(ari_results).round(4)

print("\nBOOTSTRAP ARI STABILITY ")
print(ari_df.to_string(index=False))

# Top 3 most stable by ARI
top3_ari = ari_df.nlargest(3, "ARI_mean")

print("\nTOP 3 MOST STABLE BY ARI ")
print(top3_ari.to_string(index=False))

c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\Study\Hochschule_Schmalkalden\Thesis final\workspace\Exp\.venv\lib\site-packages\sklearn\manifold\_spectral_embedding.py:328: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(
c:\Users\shetu\S


BOOTSTRAP ARI STABILITY 
    algorithm   k  ARI_mean  ARI_std  eps  threshold  min_samples
      K-Means 2.0    0.9885   0.0084  NaN        NaN          NaN
      K-Means 3.0    0.9177   0.1801  NaN        NaN          NaN
      K-Means 4.0    0.9577   0.0474  NaN        NaN          NaN
      K-Means 5.0    0.8226   0.1384  NaN        NaN          NaN
      K-Means 6.0    0.7693   0.0947  NaN        NaN          NaN
      K-Means 7.0    0.7143   0.1304  NaN        NaN          NaN
      K-Means 8.0    0.6651   0.0793  NaN        NaN          NaN
          GMM 2.0    0.9265   0.1731  NaN        NaN          NaN
          GMM 3.0    0.4896   0.2452  NaN        NaN          NaN
          GMM 4.0    0.5419   0.1714  NaN        NaN          NaN
          GMM 5.0    0.5431   0.1433  NaN        NaN          NaN
          GMM 6.0    0.4982   0.1025  NaN        NaN          NaN
          GMM 7.0    0.4558   0.0901  NaN        NaN          NaN
          GMM 8.0    0.4999   0.0629  NaN        N

In [12]:
#the top three by stability score
ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

top_3 = (
    ari_df
    .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
    .head(3)
    .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
)

print(top_3.round(4).to_string(index=False))

algorithm   k  ARI_mean  ARI_std  Stability Score
   DBSCAN NaN    1.0000   0.0000           1.0000
    BIRCH NaN    0.9987   0.0018           0.9964
    BIRCH NaN    0.9948   0.0036           0.9928


In [13]:
#the best three clustering results overall, sort primarily by ARI_mean
top_3 = ari_df.sort_values(
    ["ARI_mean", "Stability Score"],
    ascending=[False, False]
).head(3)

print(top_3.round(4).to_string(index=False))

algorithm   k  ARI_mean  ARI_std  eps  threshold  min_samples  Stability Score
   DBSCAN NaN    1.0000   0.0000  1.0        NaN          NaN           1.0000
    BIRCH NaN    0.9987   0.0018  NaN        0.2          NaN           0.9964
    BIRCH NaN    0.9948   0.0036  NaN        0.5          NaN           0.9928


In [15]:
ari_df.to_csv("updated_data/ARI_Score/ames_scaled_ari.csv", index=False)

In [14]:
#Combine all algorithm results
all_results = (
    kmean_scaled +
    gmm_scaled +
    agg_scaled +
    spec_scaled +
    dbscan_scaled +
    birch_scaled +
    optics_scaled
)

results_df = pd.DataFrame(all_results)

# Round metric values to 4 decimal places
metric_cols = ["silhouette", "davies_bouldin", "calinski_harabasz"]
results_df[metric_cols] = results_df[metric_cols].round(4)

# Columns that may exist depending on algorithm
possible_cols = ["algorithm", "k", "eps", "threshold", "min_samples","n_clusters"]

def available_cols(df, metric):
    cols = [c for c in possible_cols if c in df.columns]
    cols.append(metric)
    return cols

#  Top 3 by Silhouette (higher is better) 
top3_sil = results_df.nlargest(3, "silhouette")

print("\nTOP 3 SILHOUETTE ")
print(top3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Top 3 by Davies-Bouldin (lower is better) 
top3_db = results_df.nsmallest(3, "davies_bouldin")

print("\nTOP 3 DAVIES-BOULDIN")
print(top3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

#  Top 3 by Calinski-Harabasz (higher is better) 
top3_ch = results_df.nlargest(3, "calinski_harabasz")

print("\n TOP 3 CALINSKI-HARABASZ ")
print(top3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))

# Bottom 3 by Silhouette (lower is worse)
bottom3_sil = results_df.nsmallest(3, "silhouette")

print("\n BOTTOM 3 SILHOUETTE ")
print(bottom3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Bottom 3 by Davies-Bouldin (higher is worse) 
bottom3_db = results_df.nlargest(3, "davies_bouldin")

print("\nBOTTOM 3 DAVIES-BOULDIN ")
print(bottom3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Bottom 3 by Calinski-Harabasz (lower is worse) 
bottom3_ch = results_df.nsmallest(3, "calinski_harabasz")

print("\nBOTTOM 3 CALINSKI-HARABASZ ")
print(bottom3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))


TOP 3 SILHOUETTE 
algorithm   k  eps  threshold  min_samples  n_clusters  silhouette
   DBSCAN NaN  1.0        NaN          NaN         NaN      0.9074
   DBSCAN NaN  1.5        NaN          NaN         NaN      0.7779
   KMeans 2.0  NaN        NaN          NaN         NaN      0.1915

TOP 3 DAVIES-BOULDIN
algorithm   k  eps  threshold  min_samples  n_clusters  davies_bouldin
    BIRCH NaN  NaN        0.2          NaN      2919.0          0.0273
    BIRCH NaN  NaN        0.5          NaN      2898.0          0.0854
   DBSCAN NaN  1.0        NaN          NaN         NaN          0.1185

 TOP 3 CALINSKI-HARABASZ 
algorithm   k  eps  threshold  min_samples  n_clusters  calinski_harabasz
    BIRCH NaN  NaN        0.2          NaN      2919.0          1293.2745
   KMeans 2.0  NaN        NaN          NaN         NaN           731.2162
 Spectral 2.0  NaN        NaN          NaN         NaN           707.2466

 BOTTOM 3 SILHOUETTE 
algorithm   k  eps  threshold  min_samples  n_clusters  silho

In [15]:

# TOP 3 RESULTS FOR EACH ALGORITHM INDIVIDUALLY



all_algorithms = {
    "K-Means": kmean_scaled,
    "GMM": gmm_scaled,
    "Agglomerative": agg_scaled,
    "Spectral": spec_scaled,
    "DBSCAN": dbscan_scaled,
    "BIRCH": birch_scaled,
    "OPTICS": optics_scaled
}



for algorithm, results in all_algorithms.items():

    if len(results) == 0:
        continue

    result_df = pd.DataFrame(results)


    # Round all validation values to 4 decimal places

    result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ] = result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ].round(4)



    print("\n")
   
    print(algorithm)



 
    # Select parameter column
  

    parameter_columns = [
        "k",
        "eps",
        "threshold",
        "min_samples"
    ]


    parameter = None

    for col in parameter_columns:
        if col in result_df.columns:
            parameter = col
            break




    # TOP 3 SILHOUETTE
 

    print("\nTop 3 Silhouette Score (Higher is better)")

    top_sil = result_df.nlargest(
        3,
        "silhouette"
    )

    print(
        top_sil[
            [
                parameter,
                "silhouette"
            ]
        ].to_string(index=False)
    )




    # TOP 3 DAVIES-BOULDIN


    print("\nTop 3 Davies-Bouldin Index (Lower is better)")

    top_db = result_df.nsmallest(
        3,
        "davies_bouldin"
    )

    print(
        top_db[
            [
                parameter,
                "davies_bouldin"
            ]
        ].to_string(index=False)
    )




    # TOP 3 CALINSKI-HARABASZ


    print("\nTop 3 Calinski-Harabasz Index (Higher is better)")

    top_ch = result_df.nlargest(
        3,
        "calinski_harabasz"
    )

    print(
        top_ch[
            [
                parameter,
                "calinski_harabasz"
            ]
        ].to_string(index=False)
    )



K-Means

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1915
 3      0.1621
 4      0.1186

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.9416
 3          2.1389
 5          2.3623

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2           731.2162
 3           497.3631
 4           416.1981


GMM

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1809
 3      0.1178
 4      0.0914

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.9831
 3          2.6123
 4          2.6503

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2           667.8890
 3           435.7586
 4           306.9117


Agglomerative

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1761
 3      0.1142
 4      0.1021

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.9994
 8          2.4809
 6          2.5349

Top 3 Calinski-H